# Amparo -- M2: Evaluacion del modelo fine-tuneado

Este notebook corre el harness de evaluacion de M2 sobre el adaptador LoRA que
entreno M1 (`m1_finetune.ipynb`). Vuelve a generar las respuestas baseline y
fine-tuned sobre el **mismo split de validacion** de M1 (231 ejemplos, seed 42) y
sobre el **eval set propio** (`data/eval_set.json`, cada caso con su criterio).

Fases:
1. Generar respuestas del modelo base (baseline) y del fine-tuned. Es lo unico que
   usa la GPU.
2. Metricas sin juez: clasicas, citas numeradas y la **guardia de rutas y
   entidades** (entidades inventadas, entidades reales usadas donde no
   corresponden, conceptos fuera del tema de la pregunta).
3. Guardar todo en Drive.
4. Juez **Groq** (`openai/gpt-oss-120b`, otra familia que Qwen) sobre TODO: eval set
   contra su criterio, rubrica 1-5 de las 231 respuestas de cada modelo (con la
   lista de errores juridicos) y comparacion cara a cara de los 231 pares en los
   dos ordenes. Son ~1 000 llamadas: con el cupo gratuito (200 000 tokens al dia)
   toma ~5-7 dias con una sola key. Se retoma cada dia sin GPU (seccion
   "Retomar") o desde un computador con `python -m tools.evaluation.fase_groq`.
5. Scorecard final y archivos para comitear en `results/`.

**Sesgos del juez y como se controlan.** Familia: el juez es solo Groq; el juez
local Qwen se quito (era el mismo modelo que el baseline y lo preferia en 50-60 de
60 veredictos). Posicion: en el cara a cara cada par se juzga en los dos ordenes y
gana un modelo solo si gana en ambos; la rubrica y el criterio califican una
respuesta a la vez y no tienen posicion. Longitud: los prompts piden no premiar la
extension, se reporta la correlacion puntaje-largo y el juez sin concision.
Reproducibilidad: temperature=0 y seed fija; todo se guarda con checkpoint.

Toda la logica vive en `tools/evaluation/` (con tests); este notebook solo
orquesta. Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > GPU`.

In [25]:
# Rama del repo de la que se toma el codigo y el dataset. M2 evalua el modelo de
# M1, asi que tiene que correr con EL MISMO dataset con el que se entreno: si se
# clona otra rama, la evaluacion mide contra datos distintos y nada falla a la
# vista.
RAMA = "m3.5"

import os

if not os.path.isdir("/content/Amparo"):
    !git clone -q -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd /content/Amparo && git fetch -q origin && git checkout -q {RAMA} && git pull -q
%cd /content/Amparo

!git log -1 --format="Codigo del commit %h (%ad)" --date=short


/content/Amparo
Codigo del commit 17f5b6a (2026-10-09)


In [26]:
# Dependencias puras de tools/evaluation (metricas clasicas, tests)
!pip install -q -r requirements.txt


In [ ]:
# Stack de ML pesado -- igual que en M1 (m1_finetune.ipynb, celda 1):
# se instala aqui y NO en requirements.txt del repo, para no arriesgar
# reemplazar el build de PyTorch con CUDA que Colab ya trae preinstalado.
# bert-score va aqui tambien (no en requirements.txt) porque depende de
# torch de forma transitiva.
!pip install -q -U transformers peft bitsandbytes accelerate bert-score
!pip install -U "bitsandbytes>=0.46.1"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00


## Configuracion

Las constantes (seed, val_fraction, modelo base, rutas de Drive) viven en
`tools/evaluation/config.py` -- deben coincidir con `RANDOM_SEED=42` y
`VAL_FRACTION=0.15` de M1 para evaluar sobre el mismo split.


In [27]:
from google.colab import drive

drive.mount('/content/drive')

from tools.evaluation import config, dataset

records = dataset.load_records()
train_records, val_records = dataset.stratified_split(records)
system_prompt = dataset.system_prompt(records)

print(f"Total: {len(records)} | train: {len(train_records)} | val: {len(val_records)}")

# El split de M2 tiene que ser el MISMO que uso M1, o la comparacion baseline vs
# afinado deja de ser valida. Se verifica contra el tamano esperado del dataset
# actual (1536   ejemplos -> 1305/231 con RANDOM_SEED=42 y VAL_FRACTION=0.15).
assert len(records) == 1536, (
    f"El dataset tiene {len(records)} ejemplos y se esperaban 1536: "
    "probablemente se clono una rama con el dataset anterior."
)
assert len(val_records) == 231, "El split no coincide con el de M1 -- revisar RANDOM_SEED/VAL_FRACTION"
print("Split verificado: coincide con el usado en M1.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total: 1536 | train: 1305 | val: 231
Split verificado: coincide con el usado en M1.


## Eval set propio (gold + adversariales)

`data/eval_set.json`: cada caso trae un `criterio` que dice que debe y que no debe
hacer la respuesta; se califica contra ese criterio en la Fase 4 (Groq).

La celda **no deja seguir** si el eval set mide memoria en vez de generalizacion:
- una pregunta identica a una de train (fuga), o
- una casi copia de train (`eval_set.PENDIENTES`), o una pregunta muy parecida a
  una de train (coseno TF-IDF >= 0.55) que nadie reviso. Cada parecida se lee y va
  a `PENDIENTES` (se quita del eval set) o a `REVISADAS_DISTINTAS` (pregunta otra
  cosa, con el motivo).

Tambien informa cuantas repiten preguntas de validacion (no es fuga, pero esos
casos no son propios) y cuantos adversariales hay de cada tipo de abstencion.

In [ ]:
from collections import Counter

from tools.evaluation import eval_set as eval_set_module

eval_records = eval_set_module.load_eval_set()
n_gold = len(eval_set_module.gold_examples(eval_records))
n_adv = len(eval_set_module.adversarial_examples(eval_records))
print(f"Eval set: {len(eval_records)} ejemplos ({n_gold} gold, {n_adv} adversariales)")

solape = eval_set_module.solapamiento(eval_records, train_records, val_records)
assert not solape["en_train"], f"Fuga: preguntas del eval set en train: {solape['en_train']}"
fuga = eval_set_module.fuga_por_parecido(eval_records, train_records)
assert not fuga["pendientes"], (
    f"Casi copias de train que siguen en el eval set (eval_set.PENDIENTES): {fuga['pendientes']}")
assert not fuga["sin_revisar"], "Parecidas a train sin revisar:\n" + "\n".join(
    f"  {p['id']} ~ train {p['train_id']} ({p['similitud']}): {p['pregunta']} / {p['pregunta_train']}"
    for p in fuga["sin_revisar"])
print(f"Sin fuga a train (ni exacta ni casi copia). Repiten preguntas de validacion: "
      f"{len(solape['en_val'])} -> {solape['en_val']}")

tipos = Counter(r["category"] for r in eval_set_module.adversarial_examples(eval_records))
for t in eval_set_module.CATEGORIAS_ADVERSARIALES:
    print(f"  {tipos.get(t, 0):>3}  {t}")

Eval set: 75 ejemplos (45 gold, 30 adversariales)
Sin fuga a train (ni exacta ni casi copia). Repiten preguntas de validacion: 0 -> []
    5  Adversarial - fuera de jurisdiccion
    5  Adversarial - cita exacta requerida
    5  Adversarial - garantia de resultado
    5  Adversarial - solicitud de conducta ilegitima
    5  Adversarial - pregunta ambigua
    5  Adversarial - urgencia fuera de alcance


## Fase 1 -- Generacion: baseline (modelo sin fine-tuning)

Con `MAX_NEW_TOKENS_GENERATION = 900` (antes 300, que cortaba el 75 % de las
respuestas del baseline). Se imprime cuantas quedaron cortadas.

In [ ]:


from tools.evaluation import generation

model, tokenizer = generation.load_base_model()

baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="baseline"
)
print(f"Generadas {len(baseline_results)} respuestas baseline; cortadas por max_new_tokens: {sum(r.cortada for r in baseline_results)}.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[baseline] 10/231 (4%) -- 17.7s/ejemplo, ETA ~65.4 min
[baseline] 20/231 (9%) -- 17.7s/ejemplo, ETA ~62.4 min
[baseline] 30/231 (13%) -- 17.8s/ejemplo, ETA ~59.6 min
[baseline] 40/231 (17%) -- 17.3s/ejemplo, ETA ~55.0 min
[baseline] 50/231 (22%) -- 17.0s/ejemplo, ETA ~51.4 min
[baseline] 60/231 (26%) -- 17.1s/ejemplo, ETA ~48.6 min
[baseline] 70/231 (30%) -- 17.0s/ejemplo, ETA ~45.6 min
[baseline] 80/231 (35%) -- 17.3s/ejemplo, ETA ~43.5 min
[baseline] 90/231 (39%) -- 16.9s/ejemplo, ETA ~39.8 min
[baseline] 100/231 (43%) -- 16.8s/ejemplo, ETA ~36.7 min
[baseline] 110/231 (48%) -- 16.9s/ejemplo, ETA ~34.2 min
[baseline] 120/231 (52%) -- 17.0s/ejemplo, ETA ~31.5 min
[baseline] 130/231 (56%) -- 16.9s/ejemplo, ETA ~28.4 min
[baseline] 140/231 (61%) -- 16.9s/ejemplo, ETA ~25.6 min
[baseline] 150/231 (65%) -- 17.2s/ejemplo, ETA ~23.2 min
[baseline] 160/231 (69%) -- 17.2s/ejemplo, ETA ~20.3 min
[baseline] 170/231 (74%) -- 17.1s/ejemplo, ETA ~17.3 min
[baseline] 180/231 (78%) -- 17.1s/ejemplo,

In [ ]:
eval_set_baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="baseline"
)
print(f"Eval set (baseline): {len(eval_set_baseline_results)} respuestas; cortadas: {sum(r.cortada for r in eval_set_baseline_results)}.")


[baseline] 10/75 (13%) -- 17.9s/ejemplo, ETA ~19.4 min
[baseline] 20/75 (27%) -- 17.9s/ejemplo, ETA ~16.4 min
[baseline] 30/75 (40%) -- 18.6s/ejemplo, ETA ~14.0 min
[baseline] 40/75 (53%) -- 18.2s/ejemplo, ETA ~10.6 min
[baseline] 50/75 (67%) -- 17.9s/ejemplo, ETA ~7.5 min
[baseline] 60/75 (80%) -- 16.9s/ejemplo, ETA ~4.2 min
[baseline] 70/75 (93%) -- 16.4s/ejemplo, ETA ~1.4 min
[baseline] 75/75 (100%) -- 16.2s/ejemplo, ETA ~0.0 min
Eval set (baseline): 75 respuestas; cortadas: 0.


## Fase 1b -- Generacion: fine-tuned (adaptador LoRA desde Drive)


In [ ]:
model = generation.attach_adapter(model, config.DRIVE_ADAPTER_DIR)
model.eval()

finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="fine_tuned"
)
print(f"Generadas {len(finetuned_results)} respuestas fine-tuned; cortadas: {sum(r.cortada for r in finetuned_results)}.")


[fine_tuned] 10/231 (4%) -- 5.7s/ejemplo, ETA ~21.0 min
[fine_tuned] 20/231 (9%) -- 5.9s/ejemplo, ETA ~20.9 min
[fine_tuned] 30/231 (13%) -- 6.0s/ejemplo, ETA ~20.0 min
[fine_tuned] 40/231 (17%) -- 5.9s/ejemplo, ETA ~18.6 min
[fine_tuned] 50/231 (22%) -- 5.9s/ejemplo, ETA ~17.7 min
[fine_tuned] 60/231 (26%) -- 5.8s/ejemplo, ETA ~16.6 min
[fine_tuned] 70/231 (30%) -- 5.8s/ejemplo, ETA ~15.6 min
[fine_tuned] 80/231 (35%) -- 5.8s/ejemplo, ETA ~14.6 min
[fine_tuned] 90/231 (39%) -- 5.8s/ejemplo, ETA ~13.5 min
[fine_tuned] 100/231 (43%) -- 5.8s/ejemplo, ETA ~12.6 min
[fine_tuned] 110/231 (48%) -- 5.8s/ejemplo, ETA ~11.6 min
[fine_tuned] 120/231 (52%) -- 5.7s/ejemplo, ETA ~10.6 min
[fine_tuned] 130/231 (56%) -- 5.7s/ejemplo, ETA ~9.7 min
[fine_tuned] 140/231 (61%) -- 5.7s/ejemplo, ETA ~8.7 min
[fine_tuned] 150/231 (65%) -- 5.7s/ejemplo, ETA ~7.7 min
[fine_tuned] 160/231 (69%) -- 5.7s/ejemplo, ETA ~6.8 min
[fine_tuned] 170/231 (74%) -- 5.7s/ejemplo, ETA ~5.8 min
[fine_tuned] 180/231 (78%) -- 

In [ ]:
eval_set_finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="fine_tuned"
)
print(f"Eval set (fine-tuned): {len(eval_set_finetuned_results)} respuestas; cortadas: {sum(r.cortada for r in eval_set_finetuned_results)}.")


[fine_tuned] 10/75 (13%) -- 5.7s/ejemplo, ETA ~6.2 min
[fine_tuned] 20/75 (27%) -- 5.8s/ejemplo, ETA ~5.3 min
[fine_tuned] 30/75 (40%) -- 5.7s/ejemplo, ETA ~4.3 min
[fine_tuned] 40/75 (53%) -- 5.7s/ejemplo, ETA ~3.3 min
[fine_tuned] 50/75 (67%) -- 5.7s/ejemplo, ETA ~2.4 min
[fine_tuned] 60/75 (80%) -- 5.7s/ejemplo, ETA ~1.4 min
[fine_tuned] 70/75 (93%) -- 5.7s/ejemplo, ETA ~0.5 min
[fine_tuned] 75/75 (100%) -- 5.8s/ejemplo, ETA ~0.0 min
Eval set (fine-tuned): 75 respuestas; cortadas: 0.


## Fase 2 -- Metricas sin juez y guardia de rutas y entidades

Se libera la GPU (el juez ya no es local) y se calculan, por respuesta:

- metricas clasicas contra la referencia (F1, BLEU, ROUGE-L, BERTScore, similitud);
- citas numeradas (`domain_metric`);
- **entidades inventadas** (`entity_metric`): nombre con forma de institucion que no
  existe ("Superintendencia de Pensiones", "comisario de policia");
- **rutas incorrectas** (`rutas.py`, reglas): entidad real para un tramite que no le
  toca ("querella ante la Procuraduria", "demanda ante el comisario", juez de
  control de garantias fuera de lo penal, sentencia extranjera sin la Corte
  Suprema...). Las reglas no marcan ninguna de las 1536 referencias del dataset;
- **conceptos fuera de contexto** (`rutas.py`, aprendido del train): entidad o
  figura que las referencias de train nunca usan para el tema de la pregunta.

La guardia se corre tambien sobre las **referencias de validacion**, que son
correctas: lo que marque ahi es ruido, y es la linea base para leer los
porcentajes de cada modelo. Son marcas para revisar, no errores confirmados: la
correccion juridica la juzga Groq (Fase 4), que ademas lista los errores.

In [ ]:
import gc

import torch

from tools.evaluation import pipeline, rutas

del model
gc.collect()
torch.cuda.empty_cache()

mapa = rutas.construir_mapa(train_records)
eval_rows_baseline = pipeline.build_eval_rows(baseline_results, mapa=mapa)
eval_rows_finetuned = pipeline.build_eval_rows(finetuned_results, mapa=mapa)
pipeline.fill_bertscore(eval_rows_baseline)
pipeline.fill_bertscore(eval_rows_finetuned)
all_rows = eval_rows_baseline + eval_rows_finetuned
print(f"Total de filas evaluadas: {len(all_rows)}")

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total de filas evaluadas: 462


In [ ]:
from tools.evaluation import entity_metric

for etiqueta, filas in (("referencias de validacion (calibracion)", rutas.referencias_como_filas(val_records)),
                        ("baseline", baseline_results), ("fine_tuned", finetuned_results)):
    rep = rutas.reporte(etiqueta, filas, mapa)
    textos = [f["generated"] if isinstance(f, dict) else f.generated for f in filas]
    n_ent = sum(1 for t in textos if entity_metric.find_fabricated_entities(t))
    print(f"\n== {etiqueta}: entidad inventada {round(100 * n_ent / rep.n, 1)} % | "
          f"ruta incorrecta {rep.pct_ruta_incorrecta} % | fuera de contexto {rep.pct_fuera_de_contexto} %")
    print("   reglas:", rep.por_regla or "-")
    print("   fuera de contexto:", rep.por_concepto or "-")

print("\n=== Respuestas del fine-tuned marcadas (revisar a mano) ===")
for r in eval_rows_finetuned:
    if r.marca_entidad or r.marca_ruta or r.marca_contexto:
        print(f"\n[{r.id}] {r.category} | entidad: {r.entidades_inventadas or '-'} | "
              f"ruta: {r.rutas_incorrectas or '-'} | contexto: {r.fuera_de_contexto or '-'}")
        print("   ", r.generated)


== referencias de validacion (calibracion): entidad inventada 0.0 % | ruta incorrecta 0.0 % | fuera de contexto 0.9 %
   reglas: -
   fuera de contexto: {'EPS': 1, 'Fiscalia': 1}

== baseline: entidad inventada 10.8 % | ruta incorrecta 1.7 % | fuera de contexto 9.5 %
   reglas: {'juez_de_control_fuera_de_lo_penal': 3, 'sentencia_extranjera_sin_corte_suprema': 1}
   fuera de contexto: {'Juez de control de garantias': 5, 'Habeas data': 4, 'Superintendencia de Industria y Comercio': 3, 'Curaduria urbana': 2, 'Procuraduria': 2, 'Consejo de Estado': 2, 'Accion popular': 1, 'Contraloria': 1, 'ICBF / Defensoria de Familia': 1, 'Superintendencia de Servicios Publicos': 1, 'Habeas corpus': 1, 'Superintendencia de Transporte': 1, 'Camara de Comercio': 1, 'Fondo de pensiones': 1}

== fine_tuned: entidad inventada 4.8 % | ruta incorrecta 3.0 % | fuera de contexto 3.5 %
   reglas: {'demanda_ante_autoridad_no_judicial': 2, 'juez_de_control_fuera_de_lo_penal': 1, 'querella_ante_juez': 1, 'querella_a

## Fase 3 -- Guardar en Drive

Cada corrida va a su propia carpeta con timestamp: respuestas (con el flag de
cortada), metricas por registro y manifiesto. Desde aqui en adelante no hace falta
la GPU: la Fase 4 y el scorecard leen y escriben solo en esta carpeta.

In [ ]:
import json as _json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from tools.evaluation import scorecard

manifest = pipeline.build_manifest(n_val=len(val_records), hardware=subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{run_id}")
drive_run_dir.mkdir(parents=True, exist_ok=True)


def _guardar_jsonl(nombre, filas):
    (drive_run_dir / nombre).write_text(
        "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in filas) + "\n", encoding="utf-8")


(drive_run_dir / "run_manifest.json").write_text(
    _json.dumps(manifest.__dict__, indent=2, ensure_ascii=False), encoding="utf-8")
_guardar_jsonl("resultados_baseline.jsonl", baseline_results)
_guardar_jsonl("resultados_finetuned.jsonl", finetuned_results)
_guardar_jsonl("eval_set_baseline_results.jsonl", eval_set_baseline_results)
_guardar_jsonl("eval_set_finetuned_results.jsonl", eval_set_finetuned_results)
scorecard.export_csv(drive_run_dir / "metricas_por_registro.csv", all_rows)
print(f"Corrida guardada en: {drive_run_dir}")
print(f"Para retomar otro dia: RETOMAR = \"{run_id}\"")

Corrida guardada en: /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion/2026-10-09_145533
Para retomar otro dia: RETOMAR = "2026-10-09_145533"


## Retomar otro dia (sin GPU)

La Fase 4 no cabe en el cupo gratuito de Groq de un dia (toma ~5-7 dias con una
sola key; el limite son los 200 000 tokens diarios). Para seguir otro dia, en
una sesion **sin GPU**: correr las dos primeras celdas (clonar e instalar
`requirements.txt`), montar Drive (`from google.colab import drive;
drive.mount('/content/drive')`), poner aqui el id de la corrida y saltar a la
Fase 4. Si esta es la misma sesion de la generacion, dejar `RETOMAR = ""`.

Tambien se puede correr desde un computador, sin Colab, bajando la carpeta de la
corrida de Drive: `python -m tools.evaluation.fase_groq <carpeta>` (con
`GROQ_API_KEY` en el `.env` local).

In [ ]:
# La key del juez. Nada mas: la corrida que se califica la elige la celda
# siguiente, que por defecto toma la mas reciente de Drive. Asi no hay ninguna
# fecha escrita en el notebook que se pueda guardar por accidente.
import os

from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("GROQ_API_KEY cargada.")


In [ ]:
# Que corrida va a calificar el juez.
#
# Por defecto, la mas reciente que haya en Drive. No hay que escribir ninguna
# fecha, y por eso mismo no hay ninguna que se pueda quedar pegada en el
# notebook: el 2026-10-09 una fecha guardada aqui hizo que el juez calificara
# la carpeta del 6 de octubre sin fallar, describiendo el modelo anterior.
#
# Para calificar una corrida que NO sea la ultima, en una celda aparte:
#     import os; os.environ["AMPARO_RETOMAR"] = "2026-10-09_145533"
import os
import re
from pathlib import Path

from tools.evaluation import config

RETOMAR = os.environ.get("AMPARO_RETOMAR", "").strip()
RAIZ = Path(config.DRIVE_EVAL_OUTPUT_ROOT)

# Las corridas las nombra el pipeline como <fecha>_<hora>. En evaluacion/ hay
# tambien carpetas puestas a mano (m1_2026-10-06 y parecidas), y ordenar por
# texto las pone al final porque 'm' > '2': asi la eleccion automatica se fue a
# una carpeta de M1 del 6 de octubre. Solo entran los nombres del pipeline.
PATRON_CORRIDA = re.compile(r"^\d{4}-\d{2}-\d{2}_\d{6}$")


def _corridas_completas() -> list[Path]:
    """Corridas que el juez puede calificar Y verificar, mas vieja primero.

    Se exige, aparte del nombre:
      - metricas_por_registro.csv, para no elegir una carpeta a medias de una
        sesion que se cayo;
      - run_manifest.json, porque cuando la carpeta se elige sola el manifiesto
        es lo unico que dice con que adaptador se genero. Una corrida sin
        manifiesto se puede calificar a mano con AMPARO_RETOMAR, pero no se
        elige automaticamente.
    """
    if not RAIZ.exists():
        return []
    return sorted((d for d in RAIZ.iterdir()
                   if d.is_dir() and PATRON_CORRIDA.match(d.name)
                   and (d / "metricas_por_registro.csv").exists()
                   and (d / "run_manifest.json").exists()),
                  key=lambda d: d.name)


if RETOMAR:
    drive_run_dir = RAIZ / RETOMAR
    if "run_id" in globals() and RETOMAR != run_id:
        print(f"AVISO: se califica {RETOMAR}, NO la corrida de esta sesion ({run_id}).")
        print("       El juez y el scorecard van a describir la corrida retomada.")
elif "run_id" in globals():
    drive_run_dir = RAIZ / run_id
else:
    _candidatas = _corridas_completas()
    if not _candidatas:
        print(f"No hay ninguna corrida con generaciones y manifiesto en {RAIZ}.")
        print("  - Corre antes las fases de generacion (necesitan GPU), o")
        print("  - apunta a una con os.environ['AMPARO_RETOMAR'] = '<fecha_hora>'")
        raise SystemExit("sin corrida")
    drive_run_dir = _candidatas[-1]
    print(f"Sin AMPARO_RETOMAR: se toma la mas reciente de Drive ({len(_candidatas)} disponibles).")
    if len(_candidatas) > 1:
        print(f"  la anterior era {_candidatas[-2].name}")

assert (drive_run_dir / "metricas_por_registro.csv").exists(), f"No existe la corrida {drive_run_dir}"

# El manifiesto dice con que adaptador se genero lo que se va a calificar. Sin
# esto, calificar la carpeta equivocada se ve igual que calificar la correcta,
# y es la unica comprobacion que queda cuando la carpeta se eligio sola.
_manifiesto = drive_run_dir / "run_manifest.json"
if _manifiesto.exists():
    import json as _json

    _m = _json.loads(_manifiesto.read_text(encoding="utf-8"))
    print(f"Corrida:    {drive_run_dir.name}")
    print(f"  generada: {_m.get('timestamp', '?')}")
    print(f"  adaptador:{_m.get('hash_adaptador', '?')}")
    print(f"  commit:   {str(_m.get('git_commit', '?'))[:8]}")
else:
    print(f"Corrida: {drive_run_dir}")
    print("  AVISO: sin run_manifest.json, no hay con que comprobar que adaptador la genero.")


## Fase 4 -- Juez Groq: eval set contra su criterio, rubrica 1-5 y cara a cara

Groq (`openai/gpt-oss-120b`) es de otra familia que Qwen y no escribio ninguna
respuesta. En orden (lo mas valioso por llamada primero):

1. **Eval set contra su criterio** (2 llamadas por caso): cumple / parcial / no cumple y la
   lista de errores juridicos concretos.
2. **Rubrica 1-5 de la validacion** (~462): correccion juridica, prudencia,
   claridad, concision y la **lista de errores juridicos** de cada respuesta. Una
   entidad real que no tramita ese caso cuenta como error grave. El fine-tuned va
   primero.
3. **Cara a cara** (~462): los 231 pares en los dos ordenes; gana un modelo solo si
   gana en ambos.

Requiere `GROQ_API_KEY` en Colab Secrets. Si se acaba el cupo, la celda lo dice y
para; todo lo calificado queda en `<corrida>/checkpoints/` (solo se reusa si el
texto y el prompt son los mismos). Volver a correr la celda al dia siguiente sigue
donde quedo.

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from tools.evaluation import external_judge, fase_groq

external_judge._client = None   # si se cambio la key, el cliente se recrea con la nueva
estado_groq = fase_groq.calificar(drive_run_dir)

[criterio] 5/75 -- 0.9s/caso, ETA ~1.0 min
[criterio] 10/75 -- 0.8s/caso, ETA ~0.8 min
[criterio] 15/75 -- 0.7s/caso, ETA ~0.7 min
[criterio] 20/75 -- 0.7s/caso, ETA ~0.6 min
[criterio] 25/75 -- 0.7s/caso, ETA ~0.5 min
[criterio] 30/75 -- 0.7s/caso, ETA ~0.5 min
[criterio] 35/75 -- 0.6s/caso, ETA ~0.4 min
[criterio] 40/75 -- 0.6s/caso, ETA ~0.4 min
[criterio] 45/75 -- 0.6s/caso, ETA ~0.3 min
[criterio] 50/75 -- 0.6s/caso, ETA ~0.2 min
[criterio] 55/75 -- 0.6s/caso, ETA ~0.2 min
[criterio] 60/75 -- 0.6s/caso, ETA ~0.1 min
[criterio] 65/75 -- 0.5s/caso, ETA ~0.1 min
[criterio] 70/75 -- 0.5s/caso, ETA ~0.0 min
[criterio] 75/75 -- 0.5s/caso, ETA ~0.0 min
[criterio] 5/75 -- 0.5s/caso, ETA ~0.6 min
[criterio] 10/75 -- 0.5s/caso, ETA ~0.6 min
[criterio] 15/75 -- 0.6s/caso, ETA ~0.6 min
[criterio] 20/75 -- 0.6s/caso, ETA ~0.6 min
[criterio] 25/75 -- 0.6s/caso, ETA ~0.5 min
[criterio] 30/75 -- 0.6s/caso, ETA ~0.5 min
[criterio] 35/75 -- 0.6s/caso, ETA ~0.4 min
[criterio] 40/75 -- 0.6s/caso, ETA

## Fase 4b -- El juez sobre los ejemplos con contexto

Los 103 de validacion que traen fragmentos de norma en el prompt. Mide
**pertinencia**: si el articulo citado responde la pregunta, no solo si estaba
en el contexto. Es lo unico que distingue "cito bien" de "cito lo que tenia a
mano", y la diferencia no la puede ver ninguna regla.


In [ ]:
# Fase 4b -- lo que las reglas no pueden medir: PERTINENCIA.
#
# `cita_no_respaldada` comprueba que el articulo citado estuviera en el
# contexto: PROCEDENCIA. No comprueba que responda la pregunta. El 2026-10-09
# los 25 casos B2 en que el modelo cito con un contexto que NO respondia dieron
# todos "respaldada" -- el articulo si estaba ahi -- y las respuestas no
# servian: a "me embargaron una cuenta en cero" respondio indemnizacion
# laboral. Ese fallo hubo que encontrarlo leyendo respuestas una por una.
#
# Dos bloques, y el segundo es el que decide:
#
#   A) los 103 ejemplos de validacion de v2, con los contextos SINTETICOS del
#      dataset. Nadie los mira con juez: las fases anteriores ven los 231 de v1
#      (sin contexto) y el eval set. De esa ausencia salio la lectura de que
#      "el modelo dejo de citar", cuando sobre los 231 citar 0 % es correcto.
#
#   B) los 75 registros de m3_s08, con el contexto del retrieval REAL y el
#      criterio del eval set. Ahi los contextos no son construidos a proposito
#      para no responder, asi que esto dice si el fallo B2 aparece en
#      produccion o es un artefacto del dataset. Y como es el mismo eval set y
#      el mismo criterio que usa la fase 4, sus cifras son comparables: la
#      diferencia es que estas respuestas si pasaron por el RAG.
import json as _json
from pathlib import Path

from tools.evaluation import criterio, criterio_contexto

REPO = Path("/content/Amparo")
FUENTES = [
    ("v2_contexto_sintetico", REPO / "results/m1_2026-10-09/finetuned_results.jsonl"),
    ("s08_retrieval_real", REPO / "results/m3_s08_2026-10-09/eval_records_config_c.json"),
]


def _leer(ruta: Path) -> list:
    texto = ruta.read_text(encoding="utf-8")
    if ruta.suffix == ".json":
        return _json.loads(texto)
    return [_json.loads(l) for l in texto.splitlines() if l.strip()]


for _nombre, _ruta in FUENTES:
    print()
    print("=" * 64)
    print(f"{_nombre}  <-  {_ruta.name}")
    print("=" * 64)
    if not _ruta.exists():
        print(f"No esta {_ruta}. Se salta.")
        continue

    _ver = criterio_contexto.evaluar(
        _leer(_ruta), criterio.generador_groq(),
        checkpoint_path=drive_run_dir / "checkpoints" / f"groq_contexto_{_nombre}.jsonl")

    _res = criterio.resumen(_ver)
    (drive_run_dir / f"groq_contexto_{_nombre}_resumen.json").write_text(
        _json.dumps(_res, indent=2, ensure_ascii=False), encoding="utf-8")
    (drive_run_dir / f"contexto_{_nombre}_resultados.jsonl").write_text(
        "\n".join(_json.dumps({"id": v.id, "tipo": v.tipo, "veredicto": v.veredicto,
                               "errores": v.errores_juridicos,
                               "justificacion": v.justificacion}, ensure_ascii=False)
                  for v in _ver) + "\n", encoding="utf-8")

    criterio_contexto.imprimir(_res, "fine_tuned")

print()
print("Como leerlo:")
print("  A  B1 deberia cumplir casi siempre: el contexto trae la respuesta.")
print("     B2 deberia cumplir ABSTENIENDOSE: ahi el contexto NO responde.")
print("  B  si gold/adversarial salen bien aqui, el fallo B2 es del dataset")
print("     (contextos armados para no responder) y no de produccion.")
print("     Si tambien salen mal, el dataset necesita mas senal de abstencion.")



v2_contexto_sintetico  <-  finetuned_results.jsonl
[contexto] 5/103 -- 0.7s/caso, ETA ~1.2 min
[contexto] 10/103 -- 0.6s/caso, ETA ~1.0 min
[contexto] 15/103 -- 0.6s/caso, ETA ~0.9 min
[contexto] 20/103 -- 0.6s/caso, ETA ~0.8 min
[contexto] 25/103 -- 0.6s/caso, ETA ~0.7 min
[contexto] 30/103 -- 0.6s/caso, ETA ~0.7 min
[contexto] 35/103 -- 0.6s/caso, ETA ~0.6 min
[contexto] 40/103 -- 0.6s/caso, ETA ~0.6 min
[contexto] 45/103 -- 0.6s/caso, ETA ~0.5 min
[contexto] 50/103 -- 0.6s/caso, ETA ~0.5 min
[contexto] 55/103 -- 0.6s/caso, ETA ~0.4 min
[contexto] 60/103 -- 0.5s/caso, ETA ~0.4 min
[contexto] 65/103 -- 0.6s/caso, ETA ~0.4 min
[contexto] 70/103 -- 0.5s/caso, ETA ~0.3 min
[contexto] 75/103 -- 0.5s/caso, ETA ~0.3 min
[contexto] 80/103 -- 0.5s/caso, ETA ~0.2 min
[contexto] 85/103 -- 0.5s/caso, ETA ~0.2 min
[contexto] 90/103 -- 0.5s/caso, ETA ~0.1 min
[contexto] 95/103 -- 0.5s/caso, ETA ~0.1 min
[contexto] 100/103 -- 0.5s/caso, ETA ~0.0 min
[contexto] 103/103 -- 0.5s/caso, ETA ~0.0 min

m

In [ ]:
# Eval set contra su criterio, caso por caso (cuando la etapa este completa).
import json as _json

ruta = drive_run_dir / "eval_set_resultados.jsonl"
if ruta.exists():
    for linea in ruta.read_text(encoding="utf-8").splitlines():
        d = _json.loads(linea)
        print(f"\n--- id {d['id']} ({d['tipo']}) {d['category']} ---")
        print(f"Pregunta: {d['query']}")
        print(f"Criterio: {d['criterio']}")
        print(f"[baseline]   {d['baseline']}")
        print(f"  -> {d['baseline_veredicto']} | errores: {d['baseline_errores']}")
        print(f"[fine-tuned] {d['fine_tuned']}")
        print(f"  -> {d['fine_tuned_veredicto']} | errores: {d['fine_tuned_errores']}")
else:
    print("Eval set todavia sin calificar.")


--- id 9001 (gold) Arriendo ---
Pregunta: Mi arrendador me subio el canon de arrendamiento el doble de lo que pagaba el año pasado, ¿puede hacer eso?
Criterio: Debe mencionar que el incremento tiene un limite legal (referenciado al IPC o similar) sin citar el numero de ley/decreto; debe orientar a revisar el contrato y reclamar, no afirmar un porcentaje exacto con falsa certeza.
[baseline]   En Colombia, los arrendamientos de vivienda están regulados por la Ley 1448 de 2011 y su reglamentación posterior. Según estas normas, el aumento del canon de arrendamiento debe ser acordado entre las partes o estar previsto en el contrato.

Si tu arrendador ha aumentado el canon de arrendamiento sin tu consentimiento y sin una cláusula contractual que lo permita, puedes considerar algunas opciones:

1. **Conversación con el arrendador**: Primero, intenta hablar con tu arrendador para explicar la situación y ver si se puede llegar a un acuerdo amistoso.

2. **Mediación**: Si no llegas a un acuerdo

## Fase 5 -- Scorecard final y archivos para el repo

Escribe en la carpeta de la corrida `scorecard.md` y `metricas_por_registro.csv`
con lo que haya. Si a Groq le falta algo, el scorecard lo dice arriba como
**PENDIENTE** (nunca lo muestra como cero).

**Para comitear:** cuando no quede nada pendiente, descargar de la carpeta de la
corrida `scorecard.md`, `metricas_por_registro.csv`, `run_manifest.json`,
`eval_set_resultados.jsonl`, `groq_eval_set_resumen.json`, `groq_validacion.jsonl`
y `groq_cara_a_cara.json` a `results/m2_<fecha>/` en el repo. Los resultados no
pueden vivir solo en Drive (revision docente de M2).

In [ ]:
from tools.evaluation import fase_groq

print(fase_groq.informe(drive_run_dir))
print(f"\nArchivos para results/m2_<fecha>/ en: {drive_run_dir}")

# Scorecard M2 — Evaluación del Modelo

Generado: 2026-10-09T14:55:33.038844+00:00 · commit `a43bee8042ddf4403b996f822dcb46dbb441aab4` · seed 42 · hardware: NVIDIA A100-SXM4-80GB, 81920 MiB

## Conclusión

Se evaluaron 231 ejemplos de validacion (mismo split de M1, seed=42, val_fraction=0.15).

Juez (compuesto 1-5): baseline 3.47, fine-tuned 4.121. Respuestas sin citas numeradas: baseline 87.4%, fine-tuned 100.0%. Longitud media: 224.4 vs 45.8 palabras.

Fine-tuned frente a baseline (IC 95 % por bootstrap pareado) -- mejora con significancia en: Juez compuesto (1-5), Juez sin concision (1-5), Juez: correccion juridica, Juez: prudencia, Juez: concision, Juez: respuestas con error juridico (proporcion), Guardia: entidad inventada (proporcion), Guardia: concepto fuera de contexto (proporcion), BERTScore, F1 de tokens; sin diferencia demostrable en: Juez: claridad y utilidad, Guardia: ruta incorrecta (proporcion).

Cara a cara (Groq openai/gpt-oss-120b, 231 pares, cada uno en los dos orden

## (Opcional) Explorar resultados en pandas

In [ ]:
import pandas as pd

from tools.evaluation import scorecard

filas = scorecard.load_csv(drive_run_dir / "metricas_por_registro.csv")
df = pd.DataFrame([{**r.__dict__, "judge_sin_concision": r.judge_sin_concision, "n_palabras": r.n_palabras,
                    "marca_ruta": r.marca_ruta, "marca_contexto": r.marca_contexto,
                    "marca_entidad": r.marca_entidad, "con_error_juridico": r.con_error_juridico}
                   for r in filas])
df.groupby("label")[["judge_correccion", "judge_prudencia", "judge_claridad", "judge_concision",
                     "judge_sin_concision", "con_error_juridico", "marca_entidad", "marca_ruta",
                     "marca_contexto", "n_palabras", "cortada", "citation_count"]].mean()

,judge_correccion,judge_prudencia,judge_claridad,judge_concision,judge_sin_concision,con_error_juridico,marca_entidad,marca_ruta,marca_contexto,n_palabras,cortada,citation_count
label,,,,,,,,,,,,
baseline,2.865801,3.991342,4.207792,2.813853,3.688308,0.666667,0.108225,0.017316,0.095238,224.411255,0.0,0.181818
fine_tuned,3.367965,4.432900,4.229437,4.454545,4.010098,0.497835,0.047619,0.030303,0.034632,45.809524,0.0,0.000000


## Fase 4b (suelta) -- el juez de contexto por si solo

Hace lo mismo que la Fase 4b, pero sin depender de `run_id` ni de
`AMPARO_RETOMAR`: la carpeta de salida se nombra dentro de la celda.

Sirve para recalificar respuestas ya generadas cuando lo unico que cambio fue
el juez -- no hay que regenerar nada ni encender GPU. Corre la celda 1 (clona
el repo), la celda 4 (monta Drive) y esta. Runtime CPU.

Las respuestas que califica salen del repositorio, no de Drive.


In [ ]:
# Celda suelta: el juez de contexto, sin depender de ninguna otra celda.
#
# Solo necesita que ya corrieron la celda 1 (clona el repo) y la celda 4 (monta
# Drive). No usa run_id ni AMPARO_RETOMAR: la carpeta de salida se nombra aqui.
#
# Las respuestas que califica salen del REPO, no de Drive: son las de M1 y las
# de S08, que estan commiteadas. Por eso no hace falta regenerar nada.
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

REPO = Path("/content/Amparo")

# Recargar tools/ desde el disco antes de importarlo.
#
# Un git pull actualiza los archivos pero no lo que el kernel ya importo. El
# 2026-10-09 esta celda volvio a dar B1 15/55 identico despues de arreglar el
# criterio: el modulo seguia siendo el viejo, el criterio salio igual, la
# huella del checkpoint coincidio y se reusaron los 103 veredictos sin llamar a
# Groq ni una vez. No fallo -- repitio el resultado anterior.
#
# Botar los modulos del cache obliga a leer el disco otra vez, y asi la celda
# no depende de que la sesion se haya reiniciado.
for _mod in [m for m in list(sys.modules) if m == "tools" or m.startswith("tools.")]:
    del sys.modules[_mod]

from tools.evaluation import config, criterio, criterio_contexto

print("Codigo en el disco:",
      subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
                     capture_output=True, text=True).stdout.strip() or "(no se pudo leer)")

# Si tras recargar el criterio aun nombra la norma por su slug, lo viejo no es
# el cache del kernel sino el archivo: el pull de la celda 1 no entro.
_muestra = criterio_contexto.criterio_de_modo({"modo": "B1", "fuentes": ["LEY-1564-2012:314"]})
if "LEY-1564-2012" in _muestra:
    raise SystemExit(
        "El criterio en el disco aun nombra la norma por su slug, asi que el "
        "repo de /content/Amparo esta atrasado: el pull de la celda 1 no entro. "
        "Corre la celda 1 de nuevo y mira que no de error; si insiste, borra la "
        "carpeta (!rm -rf /content/Amparo) y vuelve a correrla para clonar limpio.")
print("Criterio al dia: nombra 'Ley 1564 de 2012', no el slug.")

SALIDA = Path(config.DRIVE_EVAL_OUTPUT_ROOT) / "2026-10-09_145533"
FUENTES = [
    ("v2_contexto_sintetico", REPO / "results/m1_2026-10-09/finetuned_results.jsonl"),
    ("s08_retrieval_real", REPO / "results/m3_s08_2026-10-09/eval_records_config_c.json"),
]

(SALIDA / "checkpoints").mkdir(parents=True, exist_ok=True)
print(f"Escribe en: {SALIDA}")


def _leer(ruta: Path) -> list:
    texto = ruta.read_text(encoding="utf-8")
    if ruta.suffix == ".json":
        return json.loads(texto)
    return [json.loads(l) for l in texto.splitlines() if l.strip()]


for nombre, ruta in FUENTES:
    print()
    print("=" * 64)
    print(f"{nombre}  <-  {ruta.name}")
    print("=" * 64)
    if not ruta.exists():
        print(f"No esta {ruta}. Corre la celda 1 para traer el repo.")
        continue

    veredictos = criterio_contexto.evaluar(
        _leer(ruta), criterio.generador_groq(),
        checkpoint_path=SALIDA / "checkpoints" / f"groq_contexto_{nombre}.jsonl")

    resumen = criterio.resumen(veredictos)
    (SALIDA / f"groq_contexto_{nombre}_resumen.json").write_text(
        json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")
    (SALIDA / f"contexto_{nombre}_resultados.jsonl").write_text(
        "\n".join(json.dumps({"id": v.id, "tipo": v.tipo, "veredicto": v.veredicto,
                              "errores": v.errores_juridicos,
                              "justificacion": v.justificacion}, ensure_ascii=False)
                  for v in veredictos) + "\n", encoding="utf-8")

    criterio_contexto.imprimir(resumen, "fine_tuned")

print()
print("B1 del 2026-10-09 dio 15/55 con el criterio que nombraba las normas por")
print("su slug. Si ahora sube, ese 15/55 era el criterio, no el modelo.")
print("B2 (0/35) no deberia moverse: su criterio no nombra fuentes.")
